# 4. Generación de indicadores — versión autónoma

Este cuaderno contiene sus propias funciones de rebanado, cálculo y exportación. No importa `config.py` ni módulos de `src/`.

> **Alcance de esta versión:** para conservar exactamente la lógica y asignación de matches existentes, procesa el universo guardado en caché (`reporte_diario`, generado el 19/10/2026); no vuelve a ejecutar los tres motores. Requiere los archivos Parquet del universo y el catálogo municipal INEGI en las rutas configuradas abajo. No se conecta a Google Drive.

Genera los mismos siete Parquet enriquecidos y el CSV de indicadores, con los nombres y formatos actuales.

## 1. Importar librerías

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
np.random.seed(0)

print(f"pandas {pd.__version__} | numpy {np.__version__}")

## 2. Configurar entorno y reproducibilidad

In [ ]:
# Localiza la raíz usando sus artefactos como marcadores; permite ejecutar desde
# la carpeta del proyecto, notebooks/ o cualquier subdirectorio.
def buscar_raiz_proyecto(inicio: Path) -> Path:
    for candidato in (inicio.resolve(), *inicio.resolve().parents):
        if (candidato / "output" / "cache" / "universo").is_dir() and (candidato / "input" / "catalogos_inegi").is_dir():
            return candidato
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. Se esperaba output/cache/universo/ "
        "e input/catalogos_inegi/ en algún directorio padre de "
        f"{inicio.resolve()}"
    )

RAIZ = buscar_raiz_proyecto(Path.cwd())
CACHE_KEY = "9132b089827c4e23"  # Snapshot reporte_diario del 19/10/2026
CACHE_DIR = RAIZ / "output" / "cache" / "universo" / CACHE_KEY
RESULTADOS_DIR = RAIZ / "output" / "Resultados_4_Generacion_Indicadores"
CATALOGO_MUNICIPIOS_FILE = RAIZ / "input" / "catalogos_inegi" / "catalogo_municipios_inegi.csv"
INDICADORES_FILE = RESULTADOS_DIR / "indicadores_selectos_desaparecidos.csv"

# Catálogo de entidades usado en los nombres y filtros del proyecto.
ENTIDADES = {
    1: "Aguascalientes", 2: "Baja California", 3: "Baja California Sur",
    4: "Campeche", 5: "Coahuila", 6: "Colima", 7: "Chiapas",
    8: "Chihuahua", 9: "Ciudad de México", 10: "Durango", 11: "Guanajuato",
    12: "Guerrero", 13: "Hidalgo", 14: "Jalisco", 15: "México",
    16: "Michoacán", 17: "Morelos", 18: "Nayarit", 19: "Nuevo León",
    20: "Oaxaca", 21: "Puebla", 22: "Querétaro", 23: "Quintana Roo",
    24: "San Luis Potosí", 25: "Sinaloa", 26: "Sonora", 27: "Tabasco",
    28: "Tamaulipas", 29: "Tlaxcala", 30: "Veracruz", 31: "Yucatán",
    32: "Zacatecas", 33: "Nacido en el Extranjero",
}

PERIODOS_TIEMPO = {
    "min-max": (None, None), "min-01ene06": (None, "2006-01-01"),
    "01ene06-max": ("2006-01-01", None), "min-01oct24": (None, "2024-10-01"),
    "01oct24-max": ("2024-10-01", None), "01ago25-max": ("2025-08-01", None),
    "01oct24-01ago25": ("2024-10-01", "2025-08-01"),
    "01ago26-max": ("2026-08-01", None), "01jul26-max": ("2026-07-01", None),
}

BASES = {
    "rn": ("rn", None), "ba": ("ba", None),
    "rn-no-ba": ("rn", ("in_banci_derivada_rnpdno_banci", False)),
    "rn-si-ba": ("rn", ("in_banci_derivada_rnpdno_banci", True)),
    "ba-no-rn": ("ba", ("in_rnpdno_derivada_rnpdno_banci", False)),
    "ba-si-rn": ("ba", ("in_rnpdno_derivada_rnpdno_banci", True)),
}
DIMENSIONES = {
    "rn": {"fecha": "fecha_hechos_modificada_derivada_rnpdno",
           "cve_ent": "idestado_derivada_rnpdno",
           "entidad": "nombre_entidad_derivada_rnpdno",
           "estatus_loc": "estatus_loc_derivada_rnpdno"},
    "ba": {"fecha": "fha_de_hchos_derivada_banci",
           "cve_ent": "cve_entidad_derivada_banci",
           "entidad": "nombre_entidad_derivada_banci",
           "estatus_loc": "estatus_loc_derivada_banci"},
}
NEGACIONES = {"sdel": "cdel", "sind": "cind", "scurp": "ccurp",
              "cisf": "sisf", "sfvac": "cfvac", "sfval": "cfval"}
CONSTANTES = {
    "rn": {},
    "ba": {"cind": True, "cfval": True, "cfvac": False,
           "fis": True, "com": False, "por": False},
}
BANDERAS_ETL = {
    "rn": {"ccurp": "curp_vic_valida_derivada_rnpdno"},
    "ba": {"ccurp": "curp_valida_derivada_banci"},
}
ESTATUS = {"all", "des", "loc"}

print(f"Raíz: {RAIZ}")
print(f"Caché seleccionado: {CACHE_DIR}")

## 3. Definir funciones auxiliares

In [ ]:
@dataclass(frozen=True, order=True)
class Slice:
    """Definición inmutable de una rebanada del universo."""
    base: str = "rn"
    periodo: str = "min-max"
    estatus: str = "all"
    subfiltro: str = "all"
    estado: Optional[int] = None

    def __post_init__(self):
        if self.base not in BASES:
            raise ValueError(f"Base desconocida: {self.base!r}; válidas: {sorted(BASES)}")
        if self.estatus not in ESTATUS:
            raise ValueError(f"Estatus desconocido: {self.estatus!r}; válidos: {sorted(ESTATUS)}")
        if self.periodo not in PERIODOS_TIEMPO:
            raise ValueError(f"Período desconocido: {self.periodo!r}; válidos: {sorted(PERIODOS_TIEMPO)}")
        if self.estado is not None and self.estado not in ENTIDADES:
            raise ValueError(f"Entidad desconocida: {self.estado!r}; válidas: {sorted(ENTIDADES)}")

    @property
    def nombre(self) -> str:
        partes = ["reg", self.base, self.periodo]
        if self.estatus != "all":
            partes.append(self.estatus)
        if self.subfiltro != "all":
            partes.append(self.subfiltro)
        if self.estado is not None:
            partes.append(f"ent{self.estado:02d}")
        return "_".join(partes)

    @property
    def entidad_texto(self) -> str:
        return ENTIDADES.get(self.estado, "Nacional") if self.estado else "Nacional"

    @property
    def padre(self) -> "Slice":
        if self.estado is not None:
            return replace(self, estado=None)
        if self.subfiltro != "all":
            tokens = self.subfiltro.split("_")[:-1]
            return replace(self, subfiltro="_".join(tokens) if tokens else "all")
        if self.estatus != "all":
            return replace(self, estatus="all")
        return replace(self, periodo="min-max")

    def con(self, **cambios) -> "Slice":
        return replace(self, **cambios)

    def por_entidad(self, cve: int) -> "Slice":
        return self.con(estado=cve)


def expandir_entidades(base: Slice, entidades=None) -> list[Slice]:
    """Construye una máscara declarativa por cada entidad del catálogo."""
    claves = entidades if entidades is not None else sorted(ENTIDADES)
    return [base.por_entidad(cve) for cve in claves]


def columna_bandera(df: pd.DataFrame, token: str) -> str:
    candidatas = [c for c in df.columns if c.startswith(f"is_{token}_derivada_")]
    if len(candidatas) != 1:
        raise KeyError(f"Bandera {token!r}: se esperaba una columna, encontradas {candidatas}")
    return candidatas[0]


def formatear_fecha_es(fecha, partes: str = "dia_mes_anio"):
    """Replica el formato español usado por los marcadores de los reportes."""
    if not fecha or not isinstance(fecha, str):
        return fecha
    trozos = fecha.split("-")
    if len(trozos) != 3:
        return fecha
    anio, mes, dia = trozos
    meses = {"01": "enero", "02": "febrero", "03": "marzo", "04": "abril",
             "05": "mayo", "06": "junio", "07": "julio", "08": "agosto",
             "09": "septiembre", "10": "octubre", "11": "noviembre", "12": "diciembre"}
    valores = {"dia": str(int(dia)), "mes": meses.get(mes, mes), "anio": anio}
    solicitadas = partes.split("_")
    desconocidas = [p for p in solicitadas if p not in valores]
    if desconocidas:
        raise ValueError(f"Partes de fecha no reconocidas: {desconocidas}")
    return " de ".join(valores[p] for p in solicitadas)


class Universo:
    """Vista local del snapshot; implementa las mismas máscaras y salidas del pipeline."""

    def __init__(self, rn: pd.DataFrame, ba: pd.DataFrame, meta: dict,
                 auditoria: Optional[pd.DataFrame] = None):
        self.rn = rn
        self.ba = ba
        self.meta = meta
        self.auditoria = auditoria
        self._cache_periodos = {}

    @classmethod
    def cargar(cls, directorio: Path) -> "Universo":
        requeridos = [directorio / n for n in ("rn.parquet", "ba.parquet", "meta.json")]
        faltantes = [str(p) for p in requeridos if not p.is_file()]
        if faltantes:
            raise FileNotFoundError("Faltan componentes del snapshot del universo: " + ", ".join(faltantes))
        rn = pd.read_parquet(requeridos[0])
        ba = pd.read_parquet(requeridos[1])
        meta = json.loads(requeridos[2].read_text(encoding="utf-8"))
        ruta_auditoria = directorio / "auditoria.csv"
        auditoria = pd.read_csv(ruta_auditoria) if ruta_auditoria.is_file() else None
        if len(rn) != meta.get("n_rn") or len(ba) != meta.get("n_ba"):
            raise ValueError("El número de filas del snapshot no coincide con meta.json; caché incompleto o inconsistente.")
        return cls(rn, ba, meta, auditoria)

    @staticmethod
    def _frame(s: Slice | str) -> str:
        return BASES[s.base][0] if isinstance(s, Slice) else s

    def columna(self, s: Slice | str, dimension: str) -> str:
        frame = self._frame(s)
        try:
            return DIMENSIONES[frame][dimension]
        except KeyError as exc:
            raise KeyError(f"Dimensión {dimension!r} no configurada para {frame!r}") from exc

    @staticmethod
    def _col_bandera(frame: str, df: pd.DataFrame, token: str) -> str:
        col = BANDERAS_ETL[frame].get(token)
        if col is not None:
            if col not in df.columns:
                raise KeyError(col)
            return col
        return columna_bandera(df, token)

    def _mascara_periodo(self, frame: str, periodo: str) -> np.ndarray:
        llave = (frame, periodo)
        if llave in self._cache_periodos:
            return self._cache_periodos[llave]
        if periodo not in PERIODOS_TIEMPO:
            raise KeyError(f"Período no definido: {periodo!r}")
        df = self.rn if frame == "rn" else self.ba
        fecha = pd.to_datetime(df[self.columna(frame, "fecha")], errors="coerce")
        inicio, fin = PERIODOS_TIEMPO[periodo]
        mask = np.ones(len(df), dtype=bool)
        nulas = fecha.isna().to_numpy()
        if inicio is not None:
            mask &= (fecha.ge(pd.Timestamp(inicio)).to_numpy() | nulas)
        if fin is not None:
            mask &= (fecha.lt(pd.Timestamp(fin)).to_numpy() | nulas)
        self._cache_periodos[llave] = mask
        return mask

    def _bandera(self, frame: str, df: pd.DataFrame, token: str, s: Slice) -> np.ndarray:
        base = NEGACIONES.get(token, token)
        if base in CONSTANTES[frame]:
            valores = np.full(len(df), CONSTANTES[frame][base], dtype=bool)
        else:
            try:
                col = self._col_bandera(frame, df, base)
            except KeyError:
                existentes = sorted(c for c in df.columns if c.startswith("is_"))
                raise KeyError(
                    f"Subfiltro {token!r} (de {s.subfiltro!r}) no disponible para "
                    f"{s.base!r}. Banderas presentes: {existentes}"
                ) from None
            valores = df[col].fillna(False).astype(bool).to_numpy()
        return ~valores if token in NEGACIONES else valores

    def _frame_y_mascara(self, s: Slice) -> tuple[pd.DataFrame, np.ndarray]:
        frame, pertenencia = BASES[s.base]
        df = self.rn if frame == "rn" else self.ba
        mask = self._mascara_periodo(frame, s.periodo).copy()
        if pertenencia is not None:
            col, valor = pertenencia
            pertenencia_mask = df[col].fillna(False).to_numpy(dtype=bool)
            mask &= pertenencia_mask if valor else ~pertenencia_mask
        if s.estatus != "all":
            col = self.columna(s, "estatus_loc")
            mask &= df[col].eq(s.estatus).fillna(False).to_numpy(dtype=bool)
        if s.subfiltro != "all":
            for token in s.subfiltro.split("_"):
                mask &= self._bandera(frame, df, token, s)
        if s.estado is not None:
            col = self.columna(s, "cve_ent")
            mask &= df[col].eq(s.estado).fillna(False).to_numpy(dtype=bool)
        return df, mask

    def contar(self, s: Slice) -> int:
        return int(self._frame_y_mascara(s)[1].sum())

    def slice(self, s: Slice) -> pd.DataFrame:
        df, mask = self._frame_y_mascara(s)
        return df.loc[mask]

    def resumen(self) -> pd.DataFrame:
        return pd.DataFrame([
            {"base": nombre, "registros": self.contar(Slice(nombre, "min-max"))}
            for nombre in BASES
        ])

    def indicadores(self, slices: Iterable[Slice], porcentajes=None, extra=None) -> dict:
        valores = {s.nombre: self.contar(s) for s in slices}
        for item in (porcentajes or []):
            s, den = item if isinstance(item, tuple) else (item, None)
            denominador = self.contar(den if den is not None else s.padre)
            n = self.contar(s)
            valores[f"pct_{s.nombre}"] = "n/d" if denominador == 0 else f"{100*n/denominador:.1f}%"
        valores["entidad_texto"] = "Nacional"
        for base, key in (("rn", "fecha_corte_rnpdno"), ("ba", "fecha_corte_banci")):
            fecha = self.meta.get(key)
            for partes in ("dia_mes_anio", "mes_anio", "anio"):
                valores[f"fha_{base}_{partes}"] = formatear_fecha_es(fecha, partes) if fecha else "Sin Fecha"
        hoy = pd.Timestamp.now().strftime("%Y-%m-%d")
        for partes in ("dia_mes_anio", "dia_mes", "mes_anio", "anio"):
            valores[f"fha_hoy_{partes}"] = formatear_fecha_es(hoy, partes)
        if extra:
            valores.update(extra)
        return valores

    def exportar_indicadores(self, slices: Iterable[Slice], destino: Path,
                             porcentajes=None, extra=None) -> pd.DataFrame:
        datos = self.indicadores(slices, porcentajes, extra)
        resultado = pd.DataFrame([{"Variable": k, "Valor": v} for k, v in datos.items()])
        destino.parent.mkdir(parents=True, exist_ok=True)
        resultado.to_csv(destino, index=False, encoding="utf-8-sig")
        print(f"📊 {len(resultado)} indicadores → {destino}")
        return resultado


## 4. Preparar datos de entrada

Se lee únicamente el snapshot guardado y el catálogo INEGI. Las funciones que filtran y transforman esos datos ya están definidas en este cuaderno.

In [ ]:
u = Universo.cargar(CACHE_DIR)

if u.meta.get("perfil") != "reporte_diario":
    raise ValueError(f"Se esperaba el snapshot reporte_diario; llegó perfil={u.meta.get('perfil')!r}")
if not CATALOGO_MUNICIPIOS_FILE.is_file():
    raise FileNotFoundError(f"No se encontró el catálogo municipal INEGI: {CATALOGO_MUNICIPIOS_FILE}")

print("Snapshot:", u.meta.get("generado"))
print("RNPDNO:", f"{len(u.rn):,}", "| BANCI:", f"{len(u.ba):,}")
print("FUBs matcheados:", f"{u.meta.get('n_fubs_matcheados', 0):,}")
print("\nResumen del universo:")
print(u.resumen().to_string(index=False))

if u.auditoria is not None:
    print("\nAuditoría de los motores:")
    print(u.auditoria.to_string(index=False))

## 5. Procesar datos con funciones internas

Los nombres canónicos, filtros por fecha/estatus/entidad y banderas son resueltos por `Slice` y `Universo`, definidos arriba.

In [ ]:
REQUERIDOS = [
    Slice("rn", "01oct24-max"),
    Slice("rn", "01oct24-max", estatus="des"),
    Slice("rn-si-ba", "01oct24-max", estatus="des"),
]

print("Conteos de rebanadas requeridas:")
for s in REQUERIDOS:
    print(f"  {s.nombre:<40} {u.contar(s):>10,}")

s_match = Slice("rn-si-ba", "01oct24-max", estatus="des")
df_match = u.slice(s_match)
df = u.slice(s_match)
print("\nDimensiones de la rebanada de desaparecidos con match:", df.shape)
display(df.head())

col_tipo_match = "tipo_match_derivada_rnpdno_banci"
conteo_match = (
    df_match[col_tipo_match].value_counts(dropna=False)
    .rename_axis("tipo_match").reset_index(name="registros")
)
conteo_match = conteo_match[conteo_match["registros"] > 0].copy()
conteo_match["pct"] = (100 * conteo_match["registros"] / len(df_match)).round(1) if len(df_match) else 0
n_fubs = df_match["fub_rnpdno"].nunique()
print(f"\n{s_match.nombre}: {len(df_match):,} registros | {n_fubs:,} FUBs únicos")
print(conteo_match.to_string(index=False))
print("\nCobertura por método (un FUB cuenta en cada método que lo sostiene):")
for metodo in ("estandar", "manual", "refinado"):
    n = df_match[col_tipo_match].astype("string").str.contains(metodo, regex=False, na=False).sum()
    porcentaje = 100 * n / len(df_match) if len(df_match) else 0
    print(f"  {metodo:<12} {n:>6,}  ({porcentaje:.1f}%)")

## 6. Generar resultados y salidas esperadas

Se conservan los nombres de las siete rebanadas, las tres columnas geográficas anexadas al final y la codificación Parquet usada actualmente.

In [ ]:
catalogo_municipios = pd.read_csv(CATALOGO_MUNICIPIOS_FILE, dtype=str, encoding="utf-8")
columnas_catalogo = {"CVEGEO", "NOM_MUN"}
if not columnas_catalogo.issubset(catalogo_municipios.columns):
    raise KeyError(f"Al catálogo INEGI le faltan columnas: {sorted(columnas_catalogo - set(catalogo_municipios.columns))}")
MAPA_MUNICIPIOS = dict(zip(catalogo_municipios["CVEGEO"], catalogo_municipios["NOM_MUN"]))


def con_claves_geo(df: pd.DataFrame) -> pd.DataFrame:
    """Añade cve_mun, cvegeo y nombre_municipio al final del DataFrame."""
    col_mun = next((c for c in ("idmunicipio_derivada_rnpdno", "id_mun_hchos_banci") if c in df.columns), None)
    if col_mun is None:
        raise KeyError(f"Sin columna de municipio. Columnas: {sorted(df.columns)}")
    sufijo = "derivada_rnpdno" if col_mun.endswith("_rnpdno") else "derivada_banci"
    col_ent = "idestado_derivada_rnpdno" if sufijo == "derivada_rnpdno" else "cve_entidad_derivada_banci"
    if col_ent not in df.columns:
        raise KeyError(f"Sin columna de entidad requerida: {col_ent}")
    salida = df.copy()
    entidad = pd.to_numeric(salida[col_ent], errors="coerce").astype("Int64")
    municipio = pd.to_numeric(salida[col_mun], errors="coerce").astype("Int64")
    completos = entidad.notna() & municipio.notna()
    cvegeo = pd.Series(pd.NA, index=salida.index, dtype="string")
    cvegeo.loc[completos] = (
        entidad.loc[completos].astype(int).map("{:02d}".format)
        + municipio.loc[completos].astype(int).map("{:03d}".format)
    )
    salida[f"cve_mun_{sufijo}"] = municipio
    salida[f"cvegeo_{sufijo}"] = cvegeo
    salida[f"nombre_municipio_{sufijo}"] = cvegeo.map(MAPA_MUNICIPIOS).astype("string")
    return salida


A_EXPORTAR = [
    Slice("rn", "min-max"),
    Slice("rn", "min-max", estatus="des"),
    Slice("rn", "min-max", estatus="loc"),
    Slice("rn", "01oct24-max", estatus="des"),
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval"),
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval_sfvac"),
    Slice("rn-si-ba", "01oct24-max", estatus="des"),
]

RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
exportados = {}
for s in A_EXPORTAR:
    df_s = con_claves_geo(u.slice(s))
    destino = RESULTADOS_DIR / f"{s.nombre}.parquet"
    df_s.to_parquet(destino, index=False, engine="pyarrow")
    exportados[s.nombre] = df_s
    sin_municipio = int(df_s["cvegeo_derivada_rnpdno"].isna().sum())
    col_fecha = "fechalocaliza_derivada_rnpdno"
    con_fecha = int(df_s[col_fecha].notna().sum()) if col_fecha in df_s else 0
    print(f"💾 {s.nombre:<24} {len(df_s):>8,} registros | "
          f"{sin_municipio:>6,} sin municipio | {con_fecha:>7,} con fecha de localización "
          f"→ {destino.name}")

columnas_vista = [
    "fub_rnpdno", "idestado_derivada_rnpdno", "nombre_entidad_derivada_rnpdno",
    "cve_mun_derivada_rnpdno", "cvegeo_derivada_rnpdno", "nombre_municipio_derivada_rnpdno",
    "fechacaptura_derivada_rnpdno", "fechalocaliza_derivada_rnpdno",
]
display(exportados["reg_rn_min-max_loc"][columnas_vista].head())

### Indicadores CSV

In [ ]:
df_indicadores = u.exportar_indicadores(REQUERIDOS, INDICADORES_FILE)
display(df_indicadores)

## 7. Validar y comprobar salidas

In [ ]:
# Conteos de referencia del snapshot seleccionado (generado 2026-10-19).
# Si se cambia el snapshot, actualizar deliberadamente este contrato de salida.
conteos_esperados = {
    "reg_rn_min-max": 412227,
    "reg_rn_min-max_des": 130458,
    "reg_rn_min-max_loc": 281769,
    "reg_rn_01oct24-max_des": 19965,
    "reg_rn-no-ba_01oct24-max_des_sfval": 6664,
    "reg_rn-no-ba_01oct24-max_des_sfval_sfvac": 5072,
    "reg_rn-si-ba_01oct24-max_des": 6966,
}
for nombre, esperado in conteos_esperados.items():
    obtenido = len(exportados[nombre])
    assert obtenido == esperado, f"{nombre}: se esperaban {esperado:,}, llegaron {obtenido:,}"

# Verifica que cada salida Parquet se pueda volver a leer y mantenga filas/esquema.
validaciones = []
for s in A_EXPORTAR:
    ruta = RESULTADOS_DIR / f"{s.nombre}.parquet"
    assert ruta.is_file(), f"No se generó {ruta}"
    releido = pd.read_parquet(ruta)
    original = exportados[s.nombre]
    assert len(releido) == len(original) == u.contar(s), f"Conteo inconsistente: {s.nombre}"
    assert list(releido.columns) == list(original.columns), f"Esquema distinto: {s.nombre}"
    validaciones.append({"archivo": ruta.name, "filas": len(releido), "columnas": len(releido.columns), "válido": True})

assert INDICADORES_FILE.is_file(), f"No se generó {INDICADORES_FILE}"
indicadores_releidos = pd.read_csv(INDICADORES_FILE, encoding="utf-8-sig")
assert list(indicadores_releidos.columns) == ["Variable", "Valor"]
assert len(indicadores_releidos) == len(df_indicadores)
for s in REQUERIDOS:
    valor = indicadores_releidos.loc[indicadores_releidos["Variable"] == s.nombre, "Valor"]
    assert len(valor) == 1 and int(valor.iloc[0]) == u.contar(s), f"Indicador no coincide: {s.nombre}"

print("Validación correcta: siete Parquet y el CSV de indicadores coinciden con los conteos de referencia.")
display(pd.DataFrame(validaciones))

# Exploración equivalente a la versión original: top 10 entidades y composición de matches.
base_entidades = Slice("rn", "01oct24-max", estatus="des")
por_entidad = pd.DataFrame([
    {"entidad": s.entidad_texto, "registros": u.contar(s)}
    for s in expandir_entidades(base_entidades)
]).sort_values("registros", ascending=False)
display(por_entidad.head(10))

df_matched = u.ba[u.ba["in_rnpdno_derivada_rnpdno_banci"].fillna(False)]
print("Composición de matches por método:")
print(df_matched["tipo_match_derivada_rnpdno_banci"].value_counts())
print("\nFUBs únicos matcheados:", f"{u.meta.get('n_fubs_matcheados', 0):,}")